# Análisis de Sentimientos en Reseñas de Películas
## Clasificación binaria con RNN, LSTM Bidireccional y LSTM + GloVe

**Dataset:** IMDb Movie Reviews (40 000 reseñas, 2 clases: Positiva / Negativa)  
**Objetivo:** Comparar tres arquitecturas de redes neuronales recurrentes para clasificación de texto,  
evaluando el impacto de la complejidad del modelo y el uso de embeddings preentrenados (GloVe).  

| Sección | Contenido |
|---|---|
| 1 | Configuración del entorno |
| 2 | Carga y exploración de datos (EDA) |
| 3 | Preprocesamiento y preparación de datos |
| 4 | Funciones de entrenamiento y evaluación |
| 5 | Modelo Base — RNN Vanilla |
| 6 | Modelo Mejorado — LSTM Bidireccional |
| 7 | Embeddings preentrenados — LSTM + GloVe |
| 8 | Comparación y evaluación de modelos |
| 9 | Inferencia en texto nuevo |

## 1. Configuración del Entorno

Se importan las librerías necesarias y se fijan las semillas de aleatoriedad (`SEED = 42`)  
para garantizar **reproducibilidad** en los experimentos.  
El dispositivo de cómputo se selecciona automáticamente: GPU si está disponible, CPU en caso contrario.

In [54]:
import pandas as pd
import numpy as np
import re
import json
import warnings
import os
import urllib.request
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de cómputo: {DEVICE}')

Dispositivo de cómputo: cuda


## 2. Carga y Exploración de Datos

### Dataset: IMDb Movie Reviews

El conjunto de datos contiene **40 000 reseñas de películas** del sitio IMDb,  
etiquetadas como `1 = Positiva` o `0 = Negativa`.  
Las clases están perfectamente balanceadas (~50 % / 50 %).

### Hallazgos del análisis exploratorio

- **Balance de clases:** 20 019 reseñas negativas / 19 981 positivas → sin necesidad de oversampling.
- **Longitud de textos:** media de ~230 palabras; rango [4, 2 470] palabras.
- **Vocabulario total:** ~20 000 términos únicos después de preprocesamiento.
- **Caracteres HTML** (`<br />`) presentes en varios textos → se eliminan en la etapa de limpieza.
- Las distribuciones de longitud son muy similares entre clases,  
  por lo que la longitud del texto por sí sola no es un predictor discriminante.

In [55]:
file_path = 'movie.csv'
df = pd.read_csv(file_path)
print(f'Shape             : {df.shape}')
print(f'Columnas          : {df.columns.tolist()}')
print(f'Valores nulos     :\n{df.isnull().sum()}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'movie.csv'

In [ ]:
print(f'Distribución de clases:\n{df["label"].value_counts()}')
print(f'\nEjemplo positivo:\n{df[df["label"]==1]["text"].iloc[0][:200]}')
print(f'\nEjemplo negativo:\n{df[df["label"]==0]["text"].iloc[0][:200]}')

: 

In [ ]:
# Métricas de longitud calculadas sobre cada reseña
df['char_count']   = df['text'].apply(len)
df['word_count']   = df['text'].apply(lambda x: len(x.split()))
df['unique_words'] = df['text'].apply(lambda x: len(set(x.lower().split())))
df['avg_word_len'] = df['text'].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
)
df['sent_label'] = df['label'].map({1: 'Positiva', 0: 'Negativa'})

print('--- Estadísticas descriptivas por clase ---')
print(
    df.groupby('sent_label')[['char_count', 'word_count', 'unique_words', 'avg_word_len']]
    .describe()
    .T
)

: 

In [ ]:
PAL = {'Positiva': '#388697', 'Negativa': '#DB162F'}
BG  = '#f8f9fa'

: 

In [ ]:
# ── Panel unificado de visualizaciones EDA ──────────────────────────────────
fig = plt.figure(figsize=(18, 14), facecolor=BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Distribución de clases
ax1 = fig.add_subplot(gs[0, 0])
vc  = df['sent_label'].value_counts()
bars = ax1.bar(vc.index, vc.values, color=[PAL[k] for k in vc.index],
               width=0.5, edgecolor='white', linewidth=1.5)
for b in bars:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
             f'{int(b.get_height()):,}', ha='center', fontweight='bold')
ax1.set_title('Distribución de Clases', fontweight='bold', pad=10)
ax1.set_ylabel('# Reseñas'); ax1.set_facecolor(BG)
ax1.spines[['top','right']].set_visible(False)

# 2. Longitud en caracteres
ax2 = fig.add_subplot(gs[0, 1])
for label, color in PAL.items():
    ax2.hist(df[df['sent_label']==label]['char_count'], bins=30,
             alpha=0.65, color=color, label=label, edgecolor='white')
ax2.set_title('Longitud (caracteres)', fontweight='bold', pad=10)
ax2.set_xlabel('Caracteres'); ax2.set_ylabel('Frecuencia')
ax2.legend(); ax2.set_facecolor(BG)
ax2.spines[['top','right']].set_visible(False)

# 3. Número de palabras
ax3 = fig.add_subplot(gs[0, 2])
for label, color in PAL.items():
    ax3.hist(df[df['sent_label']==label]['word_count'], bins=30,
             alpha=0.65, color=color, label=label, edgecolor='white')
ax3.set_title('N° de Palabras', fontweight='bold', pad=10)
ax3.set_xlabel('Palabras'); ax3.set_ylabel('Frecuencia')
ax3.legend(); ax3.set_facecolor(BG)
ax3.spines[['top','right']].set_visible(False)

# 4. Boxplot de palabras por clase
ax4 = fig.add_subplot(gs[1, 0])
data_box = [df[df['sent_label']==l]['word_count'].values for l in ['Positiva','Negativa']]
bp = ax4.boxplot(data_box, labels=['Positiva','Negativa'], patch_artist=True,
                 medianprops=dict(color='white', linewidth=2.5))
for patch, color in zip(bp['boxes'], PAL.values()):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax4.set_title('Box Plot: Palabras por Clase', fontweight='bold', pad=10)
ax4.set_ylabel('N° Palabras'); ax4.set_facecolor(BG)
ax4.spines[['top','right']].set_visible(False)

# 5. Longitud promedio de palabra
ax5 = fig.add_subplot(gs[1, 1])
for label, color in PAL.items():
    ax5.hist(df[df['sent_label']==label]['avg_word_len'], bins=20,
             alpha=0.65, color=color, label=label, edgecolor='white')
ax5.set_title('Longitud Promedio de Palabra', fontweight='bold', pad=10)
ax5.set_xlabel('Chars/palabra'); ax5.set_ylabel('Frecuencia')
ax5.legend(); ax5.set_facecolor(BG)
ax5.spines[['top','right']].set_visible(False)

# 6. Palabras únicas por reseña
ax6 = fig.add_subplot(gs[1, 2])
for label, color in PAL.items():
    ax6.hist(df[df['sent_label']==label]['unique_words'], bins=20,
             alpha=0.65, color=color, label=label, edgecolor='white')
ax6.set_title('Palabras Únicas por Reseña', fontweight='bold', pad=10)
ax6.set_xlabel('Palabras únicas'); ax6.set_ylabel('Frecuencia')
ax6.legend(); ax6.set_facecolor(BG)
ax6.spines[['top','right']].set_visible(False)

fig.suptitle('Análisis Exploratorio — IMDb Movie Reviews',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

: 

In [ ]:
# WordClouds de términos más frecuentes por clase
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)
for ax, (label, colormap) in zip(axes, [('Positiva','Blues'), ('Negativa','Reds')]):
    text = ' '.join(df[df['sent_label']==label]['text'])
    wc = WordCloud(width=700, height=350, background_color='white',
                   colormap=colormap, max_words=80, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud — Reseñas {label}s', fontweight='bold',
                 pad=10, color=PAL[label])
plt.tight_layout()
plt.show()

: 

## 3. Preprocesamiento y Preparación de Datos

### Pipeline de preprocesamiento

1. **División estratificada** 80 % entrenamiento / 20 % prueba (`stratify=label`  
   garantiza la misma proporción de clases en ambos conjuntos).
2. **`clean_text`**: minúsculas → elimina etiquetas HTML → elimina puntuación → normaliza espacios.
3. **`build_vocab`**: construye vocabulario de las 20 000 palabras más frecuentes  
   **solo del conjunto de entrenamiento** (evita data leakage).
4. **`encode`**: convierte cada texto en una secuencia de índices de longitud fija `max_len=256`;  
   los textos más cortos se rellenan con ceros (padding), los más largos se truncan.  
   `max_len=256` cubre el percentil 75 del dataset sin un costo computacional excesivo.
5. **`TextDataset` + `DataLoader`**: encapsula los datos en el formato de PyTorch;  
   `batch_size=64` equilibra velocidad de entrenamiento y uso de memoria GPU.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label'],
)

train_texts  = train_df['text'].tolist()
train_labels = train_df['label'].tolist()
test_texts   = test_df['text'].tolist()
test_labels  = test_df['label'].tolist()

print(f'Train: {len(train_texts):,} muestras | Test: {len(test_texts):,} muestras')


def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)       # elimina tags HTML (comunes en IMDb)
    text = re.sub(r'[^a-z\s]', ' ', text)   # elimina puntuación y números
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_vocab(texts, max_size=20_000):
    """Construye vocabulario sobre el conjunto de entrenamiento (sin data leakage)."""
    words = []
    for text in texts:
        words.extend(clean_text(text).split())
    freq  = Counter(words)
    vocab = {word: i+1 for i, (word, _) in enumerate(freq.most_common(max_size))}
    return vocab

vocab = build_vocab(train_texts)
print(f'Tamaño del vocabulario: {len(vocab):,} términos')


def encode(text, vocab, max_len=256):
    """Tokeniza, mapea a índices, aplica padding/truncado hasta max_len."""
    tokens = clean_text(text).split()
    idxs   = [vocab.get(token, 0) for token in tokens]  # 0 = token desconocido
    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    else:
        idxs = idxs[:max_len]
    return idxs


class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts  = texts
        self.labels = labels
        self.vocab  = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx], self.vocab), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


train_dataset = TextDataset(train_texts, train_labels, vocab)
test_dataset  = TextDataset(test_texts,  test_labels,  vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64)
print('DataLoaders creados ✓')

: 

## 4. Funciones de Entrenamiento y Evaluación

| Función | Propósito |
|---|---|
| `train_epoch` | Una pasada completa de entrenamiento; retorna loss promedio |
| `evaluate_full` | Evaluación sin gradientes; retorna loss + 4 métricas |
| `run_experiment` | Orquesta entrenamiento con early stopping y scheduler |
| `plot_training_curves` | Grafica loss, accuracy, precision/recall y F1 por época |

### Estrategia de entrenamiento

- **Optimizador:** Adam con `weight_decay=1e-5` (regularización L2 suave).
- **Scheduler:** `ReduceLROnPlateau` — reduce lr a la mitad si `val_loss` no mejora
  en 2 épocas consecutivas (`patience=2, factor=0.5`).
- **Early stopping:** detiene el entrenamiento si `val_loss` no mejora en `patience=3` épocas;
  restaura automáticamente los pesos del mejor epoch observado.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_full(model, loader, criterion, device):
    """Retorna val_loss + accuracy, precision, recall, F1."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs    = model(x_batch)
            total_loss += criterion(outputs, y_batch).item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels,  all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels,   all_preds, zero_division=0)
    f1   = f1_score(all_labels,       all_preds, zero_division=0)
    return avg_loss, acc, prec, rec, f1


def run_experiment(model, model_name, train_loader, test_loader,
                   epochs=20, patience=3, lr=3e-4):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5
    )
    history = {
        'train_loss': [], 'val_loss':      [],
        'val_acc':    [], 'val_precision': [],
        'val_recall': [], 'val_f1':        [],
    }
    best_val_loss     = float('inf')
    best_model_state  = None
    epochs_no_improve = 0

    print(f"\n{'=':=<60}")
    print(f'  Entrenando: {model_name}')
    print(f"{'=':=<60}")

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, acc, prec, rec, f1 = evaluate_full(model, test_loader, criterion, DEVICE)
        lr_antes = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        lr_despues = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(acc)
        history['val_precision'].append(prec)
        history['val_recall'].append(rec)
        history['val_f1'].append(f1)

        lr_tag = f' lr={lr_despues:.2e}' if lr_despues < lr_antes else ''
        print(
            f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | '
            f'Val Loss: {val_loss:.4f} | Acc: {acc:.4f} | '
            f'Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}{lr_tag}'
        )

        if val_loss < best_val_loss:
            best_val_loss     = val_loss
            best_model_state  = model.state_dict().copy()
            epochs_no_improve = 0
            print(f'           Mejor modelo guardado (val_loss={best_val_loss:.4f})')
        else:
            epochs_no_improve += 1
            print(f'           Sin mejora {epochs_no_improve}/{patience}')
            if epochs_no_improve >= patience:
                print(f'\nEarly stopping activado en epoca {epoch}.')
                break

    model.load_state_dict(best_model_state)
    print('\nMejor modelo restaurado')
    return history


def plot_training_curves(history, model_name, PAL, BG):
    """Grafica loss, accuracy, precision/recall y F1 a lo largo de las epocas."""
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig = plt.figure(figsize=(16, 10), facecolor=BG)
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs_range, history['train_loss'], color=PAL['Negativa'],
             linewidth=2, marker='o', markersize=4, label='Train Loss')
    ax1.plot(epochs_range, history['val_loss'],   color=PAL['Positiva'],
             linewidth=2, marker='o', markersize=4, label='Val Loss')
    ax1.set_title('Curva de Loss', fontweight='bold', pad=10)
    ax1.set_xlabel('Epoca'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.set_facecolor(BG)
    ax1.spines[['top','right']].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs_range, history['val_acc'], color=PAL['Positiva'],
             linewidth=2, marker='s', markersize=4, label='Val Accuracy')
    best_acc = max(history['val_acc'])
    best_ep  = history['val_acc'].index(best_acc) + 1
    ax2.axvline(best_ep, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax2.axhline(best_acc, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax2.annotate(f'Best: {best_acc:.4f}', xy=(best_ep, best_acc),
                 xytext=(best_ep+0.3, best_acc-0.02), fontsize=9, color='gray')
    ax2.set_title('Accuracy (Validacion)', fontweight='bold', pad=10)
    ax2.set_xlabel('Epoca'); ax2.set_ylabel('Accuracy')
    ax2.legend(); ax2.set_facecolor(BG)
    ax2.spines[['top','right']].set_visible(False)

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(epochs_range, history['val_precision'], color=PAL['Negativa'],
             linewidth=2, marker='^', markersize=4, label='Precision')
    ax3.plot(epochs_range, history['val_recall'],    color=PAL['Positiva'],
             linewidth=2, marker='v', markersize=4, label='Recall')
    ax3.set_title('Precision y Recall (Validacion)', fontweight='bold', pad=10)
    ax3.set_xlabel('Epoca'); ax3.set_ylabel('Score')
    ax3.legend(); ax3.set_facecolor(BG)
    ax3.spines[['top','right']].set_visible(False)

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(epochs_range, history['val_f1'], color='#6A0572',
             linewidth=2, marker='D', markersize=4, label='F1-Score')
    best_f1    = max(history['val_f1'])
    best_f1_ep = history['val_f1'].index(best_f1) + 1
    ax4.axvline(best_f1_ep, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax4.axhline(best_f1,    color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax4.annotate(f'Best: {best_f1:.4f}', xy=(best_f1_ep, best_f1),
                 xytext=(best_f1_ep+0.3, best_f1-0.02), fontsize=9, color='gray')
    ax4.set_title('F1-Score (Validacion)', fontweight='bold', pad=10)
    ax4.set_xlabel('Epoca'); ax4.set_ylabel('F1')
    ax4.legend(); ax4.set_facecolor(BG)
    ax4.spines[['top','right']].set_visible(False)

    fig.suptitle(f'Curvas de Entrenamiento - {model_name}',
                 fontsize=14, fontweight='bold', y=1.01)
    safe = model_name.lower().replace(' ', '_').replace('+', 'plus')
    plt.savefig(f'training_curves_{safe}.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()

: 

## 5. Modelo Base — RNN Vanilla

La RNN Vanilla sirve como **baseline**. Su flujo es:  
`embeddings → capa RNN → dropout → clasificador lineal`  
Toma únicamente el estado oculto del último paso temporal como representación de la secuencia.

### Configuración e hiperparámetros

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `embed_dim` | 200 | Dimensión estándar para corpus medianos; consistente con GloVe 200d |
| `hidden_dim` | 254 | Capacidad suficiente para capturar patrones de sentimiento |
| `dropout` | 0.3 | Regularización moderada para reducir overfitting |
| `lr` | 1e-3 | Valor estándar para Adam en RNNs simples |

### Limitación esperada

Las RNNs sufren del problema de **gradientes que desaparecen** (*vanishing gradients*)  
en secuencias largas (>100 tokens), limitando su capacidad de capturar dependencias  
de largo alcance. Esto se reflejará en métricas inferiores comparado con LSTM.

In [ ]:
class RNNClassifier(nn.Module):
    """
    RNN Vanilla para clasificación binaria de texto.
    Usa el estado oculto del último paso como representación de la secuencia completa.
    """
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=254, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.rnn       = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout   = nn.Dropout(0.3)
        self.fc        = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x        = self.embedding(x)
        out, _   = self.rnn(x)
        out      = self.dropout(out[:, -1, :])  # estado oculto del ultimo paso
        return self.fc(out)


model_rnn = RNNClassifier(vocab_size=len(vocab)).to(DEVICE)
print(model_rnn)
total_params = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)
print(f'\nParametros entrenables: {total_params:,}')

: 

In [ ]:
history_rnn = run_experiment(
    model_rnn, 'RNN Vanilla',
    train_loader, test_loader,
    epochs=20, patience=3, lr=1e-3
)
plot_training_curves(history_rnn, 'RNN Vanilla', PAL, BG)

: 

## 6. Modelo Mejorado — LSTM Bidireccional

El LSTM (*Long Short-Term Memory*) supera a la RNN Vanilla mediante su mecanismo de compuertas  
(*forget gate*, *input gate*, *output gate*), que le permite **retener selectivamente información  
a largo plazo** y mitigar el problema del gradiente que desaparece.

La versión **bidireccional** procesa la secuencia en ambas direcciones (izquierda→derecha  
y derecha→izquierda), capturando contexto global antes y después de cada palabra.  
El clasificador recibe la concatenación de ambas representaciones (`hidden_dim × 2`).

### Configuración e hiperparámetros

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `embed_dim` | 200 | Consistente con el modelo base y GloVe |
| `hidden_dim` | 256 | Mayor capacidad; la bidireccionalidad duplica la representación final |
| `n_layers` | 2 | Dos capas LSTM apiladas capturan jerarquía semántica |
| `dropout` | 0.3 | Aplicado entre capas LSTM y sobre los embeddings |
| `bidirectional` | True | Contexto forward y backward en cada token |
| `lr` | 3e-4 | Más conservador que RNN: el LSTM es más sensible a lr elevados |

In [ ]:
class LSTMClassifier(nn.Module):
    """
    LSTM Bidireccional de 2 capas para clasificación binaria.
    Concatena los estados ocultos finales de ambas direcciones como representacion.
    """
    def __init__(
        self, vocab_size, embed_dim=200, hidden_dim=256,
        output_dim=2, n_layers=2, dropout=0.3, bidirectional=True
    ):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=bidirectional,
        )
        self.dropout  = nn.Dropout(dropout)
        fc_input_dim  = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc       = nn.Linear(fc_input_dim, output_dim)

    def forward(self, x):
        embedded       = self.emb_dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden_fwd     = hidden[-2, :, :]      # ultima capa, direccion forward
        hidden_bwd     = hidden[-1, :, :]      # ultima capa, direccion backward
        hidden_cat     = torch.cat([hidden_fwd, hidden_bwd], dim=1)
        return self.fc(self.dropout(hidden_cat))


model_lstm = LSTMClassifier(vocab_size=len(vocab)).to(DEVICE)
print(model_lstm)
total_params = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
print(f'\nParametros entrenables: {total_params:,}')

: 

In [ ]:
history_lstm = run_experiment(
    model_lstm, 'LSTM Bidireccional',
    train_loader, test_loader,
    epochs=20, patience=3, lr=3e-4
)
plot_training_curves(history_lstm, 'LSTM Bidireccional', PAL, BG)

: 

## 7. Embeddings Preentrenados — LSTM + GloVe

**GloVe** (*Global Vectors for Word Representation*) son embeddings estáticos entrenados  
sobre ~840 mil millones de tokens de texto web (Common Crawl + Wikipedia).  
Codifican relaciones semánticas y sintácticas que el modelo tardaría muchas épocas en aprender  
desde cero, acelerando la convergencia y mejorando la generalización.

### Variante utilizada: `glove.6B.200d`

| Versión | Corpus | Vocabulario | Dimensión |
|---|---|---|---|
| `glove.6B.200d` | Wikipedia 2014 + Gigaword 5 | 400 000 palabras | 200 |

Se eligió `200d` para mantener la misma dimensión de embedding que los modelos anteriores  
(comparación justa) y porque coincide con los vectores públicos disponibles en Stanford NLP.

### Estrategia: fine-tuning (`freeze_embeddings=False`)

Los pesos GloVe inicializan la capa de embeddings, pero se siguen actualizando durante  
el entrenamiento para adaptarse al dominio de reseñas de cine.  
Congelarlos (`freeze_embeddings=True`) reduciría el riesgo de sobreajuste en datasets pequeños,  
pero con 32 000 ejemplos el fine-tuning suele beneficiar al modelo.

In [ ]:
glove_zip  = 'glove.6B.zip'
glove_file = 'glove.6B.200d.txt'

if not os.path.exists(glove_file):
    print('Descargando GloVe... (puede tardar unos minutos)')
    urllib.request.urlretrieve(
        'https://nlp.stanford.edu/data/glove.6B.zip', glove_zip
    )
    print('Descomprimiendo...')
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract(glove_file)
    print('GloVe listo')
else:
    print('GloVe ya existe, se omite la descarga')


def load_glove(glove_path, vocab, embed_dim=200):
    """
    Carga vectores GloVe para las palabras presentes en el vocabulario.
    Las palabras sin cobertura en GloVe quedan como vector de ceros
    y se aprenden desde cero durante el entrenamiento.
    """
    embedding_matrix = np.zeros((len(vocab) + 1, embed_dim))
    encontradas = 0
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word   = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            if word in vocab:
                embedding_matrix[vocab[word]] = vector
                encontradas += 1
    cobertura = encontradas / len(vocab) * 100
    print(f'Cobertura GloVe: {encontradas}/{len(vocab)} palabras ({cobertura:.1f}%)')
    return torch.tensor(embedding_matrix, dtype=torch.float)


glove_matrix = load_glove(glove_file, vocab, embed_dim=200)

: 

In [ ]:
class LSTMGloVe(nn.Module):
    """
    LSTM Bidireccional con embeddings inicializados desde GloVe.
    Arquitectura identica a LSTMClassifier; la diferencia esta en
    la inicializacion de la capa de embeddings con vectores preentrenados.
    """
    def __init__(
        self, vocab_size, embedding_matrix,
        embed_dim=200, hidden_dim=256, output_dim=2,
        n_layers=2, dropout=0.3, bidirectional=True,
        freeze_embeddings=False
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.embedding.weight              = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = not freeze_embeddings

        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=bidirectional,
        )
        self.dropout  = nn.Dropout(dropout)
        fc_input_dim  = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc       = nn.Linear(fc_input_dim, output_dim)

    def forward(self, x):
        embedded       = self.emb_dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden_fwd     = hidden[-2, :, :]
        hidden_bwd     = hidden[-1, :, :]
        hidden_cat     = torch.cat([hidden_fwd, hidden_bwd], dim=1)
        return self.fc(self.dropout(hidden_cat))


model_glove = LSTMGloVe(
    vocab_size        = len(vocab),
    embedding_matrix  = glove_matrix,
    freeze_embeddings = False,
).to(DEVICE)
print(model_glove)
total_params = sum(p.numel() for p in model_glove.parameters() if p.requires_grad)
print(f'\nParametros entrenables: {total_params:,}')

: 

In [ ]:
history_glove = run_experiment(
    model_glove, 'LSTM + GloVe',
    train_loader, test_loader,
    epochs=20, patience=3, lr=3e-4
)
plot_training_curves(history_glove, 'LSTM + GloVe', PAL, BG)

: 

## 8. Comparación y Evaluación de Modelos

Se comparan los tres modelos usando las métricas del **mejor epoch** (máximo F1 en validación).

- **Accuracy**: proporción de predicciones correctas.
- **Precision**: de las predicciones positivas, cuántas son realmente positivas.
- **Recall**: de los positivos reales, cuántos fueron detectados.
- **F1-Score**: media armónica de precision y recall — métrica principal dado el balance de clases.

In [ ]:
def best_metrics(history, model_name):
    best_ep = history['val_f1'].index(max(history['val_f1']))
    return {
        'Modelo'   : model_name,
        'Accuracy' : round(history['val_acc'][best_ep],       4),
        'Precision': round(history['val_precision'][best_ep], 4),
        'Recall'   : round(history['val_recall'][best_ep],    4),
        'F1-Score' : round(history['val_f1'][best_ep],        4),
        'Val Loss' : round(history['val_loss'][best_ep],      4),
    }

tabla = pd.DataFrame([
    best_metrics(history_rnn,   'RNN Vanilla'),
    best_metrics(history_lstm,  'LSTM Bidireccional'),
    best_metrics(history_glove, 'LSTM + GloVe'),
])
print('\n-- Comparacion Final de Modelos --')
print(tabla.to_string(index=False))

: 

In [ ]:
# Curvas de entrenamiento superpuestas para los tres modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)

metricas = ['val_acc',  'val_f1',   'val_loss']
titulos  = ['Accuracy', 'F1-Score', 'Val Loss']
colores  = {
    'RNN Vanilla'        : PAL['Negativa'],
    'LSTM Bidireccional' : PAL['Positiva'],
    'LSTM + GloVe'       : '#6A0572',
}

for ax, metrica, titulo in zip(axes, metricas, titulos):
    for hist, nombre in [
        (history_rnn,   'RNN Vanilla'),
        (history_lstm,  'LSTM Bidireccional'),
        (history_glove, 'LSTM + GloVe'),
    ]:
        ax.plot(
            range(1, len(hist[metrica]) + 1), hist[metrica],
            label=nombre, color=colores[nombre],
            linewidth=2, marker='o', markersize=4
        )
    ax.set_title(titulo, fontweight='bold', pad=10)
    ax.set_xlabel('Epoca')
    ax.legend()
    ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Comparacion: RNN Vanilla vs LSTM Bidireccional vs LSTM + GloVe',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

: 

In [ ]:
def evaluar_modelo(model, loader, model_name, device, PAL, BG):
    """Imprime classification report y grafica la matriz de confusion."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            outputs = model(x_batch.to(device))
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(y_batch.numpy())

    print(f"\n{'=':=<55}")
    print(f'  Classification Report - {model_name}')
    print(f"{'=':=<55}")
    print(classification_report(all_labels, all_preds,
                                 target_names=['Negativa', 'Positiva']))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=BG)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negativa','Positiva'],
                yticklabels=['Negativa','Positiva'],
                linewidths=0.5, linecolor='white',
                annot_kws={'size':14,'weight':'bold'}, ax=ax)
    ax.set_title(f'Matriz de Confusion\n{model_name}', fontweight='bold', pad=12)
    ax.set_xlabel('Prediccion', fontweight='bold')
    ax.set_ylabel('Etiqueta Real', fontweight='bold')
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / cm.sum()
    fig.text(0.5, -0.04,
             f'TP={tp}  TN={tn}  FP={fp}  FN={fn}  |  Accuracy={acc:.4f}',
             ha='center', fontsize=10, color='gray')
    plt.tight_layout()
    fname = f'confusion_matrix_{model_name.lower().replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    return all_preds, all_labels


preds_rnn,   labels_rnn   = evaluar_modelo(model_rnn,   test_loader, 'RNN Vanilla',       DEVICE, PAL, BG)
preds_lstm,  labels_lstm  = evaluar_modelo(model_lstm,  test_loader, 'LSTM Bidireccional', DEVICE, PAL, BG)
preds_glove, labels_glove = evaluar_modelo(model_glove, test_loader, 'LSTM + GloVe',       DEVICE, PAL, BG)

: 

## 9. Inferencia en Texto Nuevo

Se prueba cada modelo con cinco reseñas de ejemplo para verificar su comportamiento  
en texto no visto: dos claramente positivas, dos claramente negativas y una ambigua.

In [ ]:
def predecir(texto, model, vocab, device, max_len=256):
    """Predice el sentimiento de un texto y muestra la confianza por clase."""
    model.eval()
    tokens = encode(texto, vocab, max_len=max_len)
    x      = torch.tensor([tokens], dtype=torch.long).to(device)
    with torch.no_grad():
        output = model(x)
        prob   = torch.softmax(output, dim=1)[0]
        pred   = output.argmax(dim=1).item()
    etiqueta = 'Positiva' if pred == 1 else 'Negativa'
    print(f'  Texto      : {texto[:80]}...')
    print(f'  Sentimiento: {etiqueta}')
    print(f'  Confianza  : Positiva={prob[1]:.4f} | Negativa={prob[0]:.4f}')
    print()
    return pred


resenas = [
    'This movie was absolutely fantastic, the acting was outstanding!',
    'Terrible film, complete waste of time. The plot made no sense at all.',
    'It was okay, nothing special but not bad either.',
    'One of the best movies I have ever seen in my entire life!',
    'I fell asleep halfway through, incredibly boring and predictable.',
]

for model_obj, nombre in [
    (model_rnn,   'RNN Vanilla'),
    (model_lstm,  'LSTM Bidireccional'),
    (model_glove, 'LSTM + GloVe'),
]:
    print(f"\n{'=':=<55}")
    print(f'  Inferencia - {nombre}')
    print(f"{'=':=<55}")
    for r in resenas:
        predecir(r, model_obj, vocab, DEVICE)

: 

In [ ]:
# Guardar pesos de los tres modelos y el vocabulario compartido
torch.save(model_rnn.state_dict(),   'model_rnn.pth')
torch.save(model_lstm.state_dict(),  'model_lstm.pth')
torch.save(model_glove.state_dict(), 'model_lstm_glove.pth')

with open('vocab.json', 'w') as f:
    json.dump(vocab, f)

print('Modelos guardados: model_rnn.pth, model_lstm.pth, model_lstm_glove.pth')
print('Vocabulario guardado: vocab.json')

: 